# Módulo 9 · Clase 3 — Almacenamiento SQL

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | SQLite con `sqlite3` — SQL puro |
| 2 | SQLAlchemy — ORM de Python |
| 3 | Cargar datos procesados en la BD |
| 4 | Consultas analíticas con SQL |
| 5 | Integración pandas ↔ SQL |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = 'data/db/vinos.db'
Path('data/db').mkdir(parents=True, exist_ok=True)

# Cargar datos procesados
df = pd.read_parquet('data/processed/vinos_procesado.parquet')
print(f"Dataset cargado: {df.shape}")
df.head(3)

---
## 1. SQLite con `sqlite3`

`sqlite3` es la biblioteca estándar de Python para bases de datos SQL. No necesitas instalar nada: SQLite guarda toda la BD en un único archivo `.db`.

In [ ]:
# Crear conexión y tabla
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# DROP + CREATE para empezar limpio
cursor.execute("DROP TABLE IF EXISTS vinos")

cursor.execute("""
CREATE TABLE vinos (
    id                     INTEGER PRIMARY KEY AUTOINCREMENT,
    fixed_acidity          REAL,
    volatile_acidity       REAL,
    citric_acid            REAL,
    residual_sugar         REAL,
    chlorides              REAL,
    free_sulfur_dioxide    REAL,
    total_sulfur_dioxide   REAL,
    density                REAL,
    pH                     REAL,
    sulphates              REAL,
    alcohol                REAL,
    so2_ratio              REAL,
    acid_index             REAL,
    density_diff           REAL,
    tipo_encoded           INTEGER,
    quality                INTEGER,
    buen_vino              INTEGER,
    tipo                   TEXT
)
""")

conn.commit()
print("✓ Tabla 'vinos' creada en", DB_PATH)

In [ ]:
# Insertar datos fila por fila (para entender el mecanismo)
# En producción usaríamos executemany o to_sql

COLUMNAS = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'so2_ratio', 'acid_index', 'density_diff',
    'tipo_encoded', 'quality', 'buen_vino', 'tipo'
]

# Insertar primeras 10 filas manualmente para ver el mecanismo
placeholders = ', '.join(['?'] * len(COLUMNAS))
sql_insert = f"INSERT INTO vinos VALUES (NULL, {placeholders})"

for _, row in df.head(10).iterrows():
    valores = tuple(row[c] for c in COLUMNAS)
    cursor.execute(sql_insert, valores)

conn.commit()
print("10 filas insertadas una a una")

# Verificar
cursor.execute("SELECT COUNT(*) FROM vinos")
print(f"Filas en BD: {cursor.fetchone()[0]}")

# executemany es MÁS EFICIENTE para muchas filas
cursor.execute("DELETE FROM vinos")  # limpiar para insertar todo
filas = [tuple(row[c] for c in COLUMNAS) for _, row in df.iterrows()]
cursor.executemany(sql_insert, filas)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM vinos")
print(f"Filas tras executemany: {cursor.fetchone()[0]}")

In [ ]:
# Consultas SQL básicas

# SELECT básico
cursor.execute("SELECT id, tipo, quality, alcohol FROM vinos LIMIT 5")
print("SELECT básico:")
for fila in cursor.fetchall():
    print(f"  {fila}")

# WHERE + ORDER BY
cursor.execute("""
    SELECT tipo, ROUND(AVG(quality), 2) AS calidad_media, COUNT(*) AS n
    FROM vinos
    GROUP BY tipo
    ORDER BY calidad_media DESC
""")
print("\nCalidad media por tipo:")
for fila in cursor.fetchall():
    print(f"  {fila}")

# Subquery: vinos con calidad superior a la media de su tipo
cursor.execute("""
    SELECT v.tipo, v.quality, v.alcohol
    FROM vinos v
    WHERE v.quality > (
        SELECT AVG(v2.quality) FROM vinos v2 WHERE v2.tipo = v.tipo
    )
    LIMIT 5
""")
print("\nVinos sobre la media de su tipo:")
for fila in cursor.fetchall():
    print(f"  {fila}")

---
## 2. SQLAlchemy — ORM de Python

In [ ]:
from sqlalchemy import create_engine, text, Column, Integer, Float, String
from sqlalchemy.orm import DeclarativeBase, Session

# Crear engine (la URL sqlite:///ruta es relativa al directorio actual)
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

print(f"✓ Engine conectado a {DB_PATH}")
print(f"  Dialect: {engine.dialect.name}")

# Verificar tablas existentes
with engine.connect() as c:
    resultado = c.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    tablas = [row[0] for row in resultado]
    print(f"  Tablas: {tablas}")

In [ ]:
# Definir modelo ORM
class Base(DeclarativeBase):
    pass

class Prediccion(Base):
    """Tabla para guardar predicciones del modelo en producción."""
    __tablename__ = 'predicciones'

    id              = Column(Integer, primary_key=True, autoincrement=True)
    timestamp       = Column(String)   # ISO 8601
    tipo_vino       = Column(String)
    alcohol         = Column(Float)
    pH              = Column(Float)
    quality_real    = Column(Integer)
    quality_pred    = Column(Float)
    buen_vino_pred  = Column(Integer)
    confianza       = Column(Float)
    modelo_version  = Column(String)

    def __repr__(self):
        return f"<Prediccion id={self.id} pred={self.quality_pred:.1f} real={self.quality_real}>"

# Crear la tabla en la BD
Base.metadata.create_all(engine)
print("✓ Tabla 'predicciones' creada")

In [ ]:
from datetime import datetime
import random

# Insertar predicciones simuladas (se usarán en C5 con el modelo real)
random.seed(42)
predicciones_demo = []
for i, (_, row) in enumerate(df.head(50).iterrows()):
    ruido = random.gauss(0, 0.5)
    pred_q = float(row['quality']) + ruido
    pred_b = int(pred_q >= 7)
    predicciones_demo.append(Prediccion(
        timestamp      = datetime.now().isoformat(),
        tipo_vino      = row['tipo'],
        alcohol        = float(row['alcohol']),
        pH             = float(row['pH']),
        quality_real   = int(row['quality']),
        quality_pred   = round(pred_q, 2),
        buen_vino_pred = pred_b,
        confianza      = round(random.uniform(0.6, 0.95), 2),
        modelo_version = 'v1.0-demo'
    ))

with Session(engine) as session:
    session.add_all(predicciones_demo)
    session.commit()

print(f"✓ {len(predicciones_demo)} predicciones demo insertadas")

---
## 3. Integración pandas ↔ SQL

In [ ]:
# pd.read_sql + pd.to_sql — los métodos más cómodos

# Leer toda la tabla
df_desde_bd = pd.read_sql('SELECT * FROM vinos LIMIT 5', engine)
print("Primeras filas desde BD:")
print(df_desde_bd[['tipo','quality','alcohol','pH']].to_string(index=False))

# Consulta analítica directamente como DataFrame
df_resumen = pd.read_sql("""
    SELECT
        tipo,
        quality,
        COUNT(*)                     AS n_vinos,
        ROUND(AVG(alcohol), 2)       AS alcohol_medio,
        ROUND(AVG(pH), 2)            AS pH_medio,
        ROUND(AVG(sulphates), 3)     AS sulfatos_medios
    FROM vinos
    GROUP BY tipo, quality
    ORDER BY tipo, quality
""", engine)

print("\nResumen por tipo y calidad:")
print(df_resumen.to_string(index=False))

In [ ]:
# to_sql — guardar un DataFrame directamente en una tabla nueva

df_stats = df.groupby(['tipo', 'quality']).agg(
    n=('quality', 'count'),
    alcohol_medio=('alcohol', 'mean'),
    pH_medio=('pH', 'mean'),
).reset_index().round(3)

df_stats.to_sql('resumen_calidad', engine,
                if_exists='replace',  # replace/append/fail
                index=False)

print("Tabla 'resumen_calidad' guardada en BD")

# Verificar
df_check = pd.read_sql('SELECT * FROM resumen_calidad', engine)
print(df_check)

---
## 4. Consultas analíticas avanzadas

In [ ]:
import matplotlib.pyplot as plt

# Window functions en SQLite (disponibles desde v3.25)
df_window = pd.read_sql("""
    SELECT
        tipo,
        quality,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY tipo), 2) AS pct_en_tipo
    FROM vinos
    GROUP BY tipo, quality
    ORDER BY tipo, quality
""", engine)

print("Distribución porcentual de calidad por tipo:")
print(df_window.to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
for tipo in df_window['tipo'].unique():
    sub = df_window[df_window['tipo'] == tipo]
    ax.plot(sub['quality'], sub['pct_en_tipo'], marker='o', label=tipo, linewidth=2)

ax.set_xlabel('Calidad'); ax.set_ylabel('% dentro del tipo')
ax.set_title('Distribución de calidad por tipo de vino')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analizar las predicciones demo
df_pred = pd.read_sql("""
    SELECT
        tipo_vino,
        COUNT(*)                                          AS n,
        ROUND(AVG(ABS(quality_real - quality_pred)), 3)  AS mae,
        ROUND(AVG(CASE WHEN buen_vino_pred = (quality_real >= 7)
                       THEN 1 ELSE 0 END), 3)            AS accuracy,
        ROUND(AVG(confianza), 3)                         AS confianza_media
    FROM predicciones
    GROUP BY tipo_vino
""", engine)

print("Métricas de predicciones demo por tipo:")
print(df_pred.to_string(index=False))

# Cerrar conexión sqlite3 manual
conn.close()
print("\n✓ Conexión sqlite3 cerrada")

---
## Práctica

### Ejercicio 1 — Top 10 mejores vinos

**Enunciado:** Escribe una consulta SQL que devuelva los 10 vinos con mayor calidad.
En caso de empate en calidad, ordénalos por alcohol de mayor a menor.
Muestra: id, tipo, quality, alcohol, pH.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine('sqlite:///data/db/vinos.db')

df_top = pd.read_sql("""
    SELECT id, tipo, quality, ROUND(alcohol, 2) AS alcohol, ROUND(pH, 2) AS pH
    FROM vinos
    ORDER BY quality DESC, alcohol DESC
    LIMIT 10
""", engine)

print("Top 10 mejores vinos:")
print(df_top.to_string(index=False))
```
</details>

---
### Ejercicio 2 — Correlación SQL vs pandas

**Enunciado:** En pandas calculamos correlaciones con `.corr()`. En SQL puro también es posible.

Fórmula de Pearson: `r = (n·Σxy - Σx·Σy) / √[(n·Σx² - (Σx)²)·(n·Σy² - (Σy)²)]`

1. Escribe una consulta SQL que calcule la correlación entre `alcohol` y `quality`
2. Compara el resultado con el que da `df[['alcohol','quality']].corr()`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine('sqlite:///data/db/vinos.db')

# Correlación en SQL
with engine.connect() as conn:
    resultado = conn.execute(text("""
        SELECT
            (COUNT(*) * SUM(alcohol * quality) - SUM(alcohol) * SUM(quality)) /
            SQRT(
                (COUNT(*) * SUM(alcohol*alcohol) - SUM(alcohol)*SUM(alcohol)) *
                (COUNT(*) * SUM(quality*quality) - SUM(quality)*SUM(quality))
            ) AS correlacion_pearson
        FROM vinos
    """))
    r_sql = resultado.fetchone()[0]

# Correlación en pandas
df = pd.read_parquet('data/processed/vinos_procesado.parquet')
r_pandas = df['alcohol'].corr(df['quality'])

print(f"Correlación alcohol-quality:")
print(f"  SQL:    {r_sql:.6f}")
print(f"  pandas: {r_pandas:.6f}")
print(f"  Diferencia: {abs(r_sql - r_pandas):.2e}")
```
</details>

---
### Ejercicio 3 — Índice de rendimiento

**Enunciado:** Crea un índice en la columna `quality` y compara el tiempo de una consulta con y sin índice:

1. `EXPLAIN QUERY PLAN SELECT ... WHERE quality = 8` antes del índice
2. `CREATE INDEX idx_quality ON vinos(quality)`
3. Repite el EXPLAIN y observa el cambio
4. Mide el tiempo con `%timeit` en un SELECT con filtro

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import sqlite3

conn = sqlite3.connect('data/db/vinos.db')
cursor = conn.cursor()

# Plan de consulta SIN índice
cursor.execute("DROP INDEX IF EXISTS idx_quality")
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM vinos WHERE quality = 8")
print("Sin índice:", cursor.fetchall())

# Crear índice
cursor.execute("CREATE INDEX IF NOT EXISTS idx_quality ON vinos(quality)")
conn.commit()

# Plan de consulta CON índice
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM vinos WHERE quality = 8")
print("Con índice:", cursor.fetchall())

conn.close()

# Medir tiempo
import timeit
def query_sin_indice():
    c = sqlite3.connect('data/db/vinos.db')
    c.execute("SELECT * FROM vinos WHERE quality = 8")
    c.fetchall(); c.close()

t = timeit.timeit(query_sin_indice, number=100)
print(f"\n100 consultas: {t:.3f}s ({t/100*1000:.2f}ms por consulta)")
print("Nota: con SQLite en local la diferencia es pequeña —"
      " en PostgreSQL con millones de filas es enorme")
```
</details>

---
### Ejercicio 4 — Logger de predicciones

**Enunciado:** Crea una función `log_prediccion(engine, datos_entrada, prediccion)` que:

1. Inserte una nueva fila en la tabla `predicciones`
2. Devuelva el `id` asignado
3. Incluya un campo `timestamp` con la fecha/hora actual
4. Sea segura ante entradas erróneas (try/except + rollback)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from sqlalchemy import create_engine, text
from datetime import datetime

engine = create_engine('sqlite:///data/db/vinos.db')

def log_prediccion(engine, datos_entrada: dict, prediccion: dict) -> int | None:
    """
    Registra una predicción en la BD y devuelve el id asignado.
    datos_entrada: {'tipo_vino', 'alcohol', 'pH'}
    prediccion:    {'quality_real', 'quality_pred', 'buen_vino_pred', 'confianza', 'modelo_version'}
    """
    sql = text("""
        INSERT INTO predicciones
            (timestamp, tipo_vino, alcohol, pH,
             quality_real, quality_pred, buen_vino_pred, confianza, modelo_version)
        VALUES
            (:timestamp, :tipo_vino, :alcohol, :pH,
             :quality_real, :quality_pred, :buen_vino_pred, :confianza, :modelo_version)
    """)
    params = {
        'timestamp': datetime.now().isoformat(),
        **datos_entrada,
        **prediccion
    }
    try:
        with engine.begin() as conn:  # commit automático al salir
            resultado = conn.execute(sql, params)
            return resultado.lastrowid
    except Exception as e:
        print(f"Error al insertar predicción: {e}")
        return None

# Test
nuevo_id = log_prediccion(
    engine,
    datos_entrada={'tipo_vino': 'blanco', 'alcohol': 12.5, 'pH': 3.2},
    prediccion={'quality_real': 7, 'quality_pred': 6.8,
                'buen_vino_pred': 1, 'confianza': 0.82, 'modelo_version': 'v1.0'}
)
print(f"Predicción registrada con id={nuevo_id}")
```
</details>

---
### Ejercicio 5 (reto) — Dashboard SQL en pandas

**Enunciado:** Crea un dashboard de texto con 5 métricas de la BD:

1. Total de vinos por tipo
2. Distribución de calidades (histograma de texto: `★★★★` = proporción)
3. Top 3 features más correlacionadas con `quality` (usa SQL para calcular)
4. Comparativa de medias para buenos vs no buenos vinos
5. Últimas 5 predicciones registradas

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
## Resumen

```python
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import DeclarativeBase, Session

# sqlite3 puro
conn   = sqlite3.connect('data/db/mi_bd.db')
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS tabla (id INTEGER PRIMARY KEY, valor REAL)")
cursor.executemany("INSERT INTO tabla VALUES (NULL, ?)", [(1.0,), (2.0,)])
conn.commit()
conn.close()

# SQLAlchemy
engine = create_engine('sqlite:///data/db/mi_bd.db')
with engine.connect() as c:
    rows = c.execute(text("SELECT * FROM tabla")).fetchall()

# pandas ↔ SQL
df = pd.read_sql("SELECT * FROM tabla WHERE valor > 1", engine)
df.to_sql('tabla_nueva', engine, if_exists='replace', index=False)
```

## ¿Qué viene en la próxima clase?
En la **Clase 4** entrenaremos los primeros modelos de Machine Learning con **Scikit-Learn**: desde un baseline hasta un RandomForest con GridSearchCV, usando los datos ya almacenados en la BD.